# Prompt Chaining example

In [15]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from IPython.display import Markdown
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [ ]:
# -----------------------------------------------------------
# STEP 1 — Initialize the LLM model
# -----------------------------------------------------------
# Here we create an instance of the Gemini chat model through
# the LangChain wrapper.
#
# model="gemini-2.5-flash"
#   → a fast and efficient Gemini model suitable for most
#     conversational or question-answering tasks.
#
# temperature=0.7
#   → controls randomness of the generated output.
#   → lower values (0.2–0.3) produce more deterministic answers
#   → higher values (0.7–1.0) produce more creative responses.
#
# This model object will later be used inside a LangGraph node
# to generate responses from the LLM.

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [ ]:
# -----------------------------------------------------------
# STEP 2 — Define the structure of the graph state
# -----------------------------------------------------------
# In a LangGraph workflow, all nodes communicate through a
# shared state object (a dictionary).
#
# TypedDict is used to formally define the expected structure
# of this state. This improves:
# - code clarity
# - type safety
# - easier debugging
#
# Fields in this workflow:
#
# title: the blog topic provided by the user
#
# outline: the structured outline generated by the LLM
#
# content: the full blog article generated from the outline

# define the state
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [ ]:
# -----------------------------------------------------------
# STEP 3.1 — Node 1: Generate the blog outline
# -----------------------------------------------------------
# This node performs the first stage of the pipeline:
# turning a blog title into a structured outline.
#
# Why this node exists:
# Instead of generating the entire blog directly, we first
# create an outline so the final content is more structured
# and logically organized.
#
# This mirrors how humans typically write articles:
# idea → outline → full draft

def create_outline(state: BlogState) -> BlogState:
    # fetch title
    title = state['title']

    # call llm to gen outline
    prompt = f'Generate a detailed outline for a blog on the topic: {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline
    return state


# -----------------------------------------------------------
# STEP 3.2 — Node 2: Generate the blog content
# -----------------------------------------------------------
# This node expands the previously generated outline into
# a full blog article.
#
# Why this step is separated:
# - It improves structure and coherence
# - The LLM can focus on expanding each section properly
# - It mirrors real-world writing workflows

def write_content(state: BlogState) -> BlogState:
    # fetch title and outline
    title = state['title']
    outline = state['outline']

    # call llm to gen content
    prompt = f'Write a blog post based on the following title and outline.\n\nTitle: {title}\n\nOutline: {outline}'
    content = model.invoke(prompt).content

    # update state
    state['content'] = content
    return state

In [ ]:
# -----------------------------------------------------------
# STEP 4 — Create the LangGraph workflow
# -----------------------------------------------------------
# StateGraph defines a directed graph where nodes operate
# on a shared state object.
#
# BlogState specifies the structure of the state used
# throughout the workflow.

# define the graph
graph = StateGraph(BlogState)

# add nodes
graph.add_node('create_outline', create_outline)
graph.add_node('write_content', write_content)

# add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'write_content')
graph.add_edge('write_content', END)

# compile
workflow = graph.compile()

In [ ]:
# -----------------------------------------------------------
# STEP 5 — Execute the workflow
# -----------------------------------------------------------
# invoke() runs the graph from START to END.
#
# Execution sequence:
#
# 1. create_outline generates blog structure
# 2. write_content expands outline into full article
# 3. The final state is returned

initial_state = {'title': "The Future of AI in Agriculture"}

final_state = workflow.invoke(initial_state)

In [ ]:
# -----------------------------------------------------------
# STEP 6 — Display the generated outline
# -----------------------------------------------------------
# Markdown() renders the text using Markdown formatting
# inside a Jupyter notebook.
#
# This makes headings and bullet points more readable.

Markdown(final_state['outline'])

Here's a detailed outline for a blog post on "The Future of AI in Agriculture," designed to be comprehensive, engaging, and informative.

---

## Blog Post Title Options:
*   Harvesting Tomorrow: How AI is Revolutionizing the Future of Agriculture
*   From Soil to Silicon: The AI-Powered Transformation of Farming
*   Smart Farms, Smarter Food: Unpacking the Future of AI in Agriculture
*   Beyond the Tractor: AI's Role in Feeding a Growing World

---

## Blog Post Outline: The Future of AI in Agriculture

### I. Introduction (Approx. 150-200 words)

*   **A. Catchy Hook:** Start with a compelling statistic or a thought-provoking question about global food demand, climate change, or the challenges faced by modern agriculture.
    *   *Example:* "By 2050, the world will need to feed nearly 10 billion people. How can we possibly meet this demand while grappling with climate change, resource scarcity, and labor shortages?"
*   **B. Context & Current State (Briefly):** Acknowledge the traditional importance of agriculture and the increasing pressures it faces.
    *   Mention existing agricultural challenges: dwindling arable land, water scarcity, pest resistance, unpredictable weather, labor costs.
*   **C. Introducing AI as the Game Changer:** Position Artificial Intelligence as a pivotal solution to these challenges, ushering in a new era of "smart farming."
    *   Briefly define what AI means in this context (machine learning, computer vision, robotics, data analytics).
*   **D. Thesis Statement / What Readers Will Learn:** Clearly state the blog's purpose – to explore the transformative potential of AI across various facets of agriculture, from farm management to global food security.

### II. The Current Landscape: AI's Footprint in Today's Agriculture (Approx. 100 words)

*   **A. Early Adoptions:** Briefly mention where AI is already making an impact.
    *   Basic weather forecasting models
    *   Early drone mapping for field analysis
    *   Automated irrigation systems (though not always "AI" in a complex sense, they lay groundwork)
*   **B. Setting the Stage for the Future:** Emphasize that what we see now is just the tip of the iceberg.

### III. Key Pillars of AI's Future in Agriculture (Main Body - Approx. 800-1000 words)

*   **A. Precision Agriculture & Resource Optimization**
    *   **1. Smart Irrigation Systems:**
        *   *How AI helps:* Analyzing soil moisture, weather forecasts, crop type, and growth stage to deliver precise water amounts.
        *   *Benefits:* Significant water savings, reduced runoff, optimized crop health.
        *   *Technologies:* IoT sensors, machine learning algorithms.
    *   **2. Variable Rate Application (Fertilizers & Pesticides):**
        *   *How AI helps:* Drones and sensors collect hyper-local data on plant health and nutrient deficiencies. AI algorithms then dictate exact amounts and locations for application.
        *   *Benefits:* Reduced chemical use, lower environmental impact, cost savings, higher yields.
        *   *Technologies:* Computer vision, spectral imaging, GPS-guided machinery.
    *   **3. Automated Weed Detection & Removal:**
        *   *How AI helps:* AI-powered robots identify weeds versus crops with high accuracy and apply targeted herbicides or mechanical removal.
        *   *Benefits:* Drastically reduced herbicide use, less labor, healthier crops.
        *   *Technologies:* Robotics, computer vision, deep learning.

*   **B. Predictive Analytics & Crop Monitoring**
    *   **1. Disease and Pest Detection:**
        *   *How AI helps:* Drones, ground sensors, and cameras capture images/data. AI models analyze patterns to identify early signs of disease or pest infestation.
        *   *Benefits:* Early intervention, preventing widespread crop loss, targeted treatments.
        *   *Technologies:* Computer vision, machine learning, satellite imagery.
    *   **2. Yield Prediction & Crop Health Forecasting:**
        *   *How AI helps:* Integrating historical data, weather patterns, soil conditions, and sensor data to accurately predict yield and monitor crop stress levels.
        *   *Benefits:* Better planning for harvest, logistics, and market pricing; proactive problem-solving.
        *   *Technologies:* Big data analytics, predictive modeling, remote sensing.

*   **C. Automated Farming & Robotics**
    *   **1. Autonomous Tractors & Farm Machinery:**
        *   *How AI helps:* Self-driving tractors and harvesters performing tasks with minimal human oversight.
        *   *Benefits:* 24/7 operation, increased efficiency, reduced labor costs, enhanced precision.
        *   *Technologies:* GPS, LiDAR, computer vision, advanced robotics.
    *   **2. Harvesting & Planting Robots:**
        *   *How AI helps:* Robots designed to delicately pick ripe fruits/vegetables or precisely plant seeds, reducing manual labor and damage.
        *   *Benefits:* Addresses labor shortages, reduces waste, improves consistency.
        *   *Technologies:* Robotic arms, grippers, computer vision for ripeness detection.

*   **D. AI in Livestock Management**
    *   **1. Individual Animal Monitoring:**
        *   *How AI helps:* Wearable sensors and computer vision track individual animal health, behavior (e.g., lameness, estrus, feeding patterns), and growth.
        *   *Benefits:* Early detection of illness, improved animal welfare, optimized feeding, increased productivity.
        *   *Technologies:* IoT sensors, facial recognition for animals, behavioral analytics.
    *   **2. Automated Feeding & Environment Control:**
        *   *How AI helps:* AI-driven systems dispense precise feed portions based on individual animal needs and optimize barn conditions (temperature, ventilation).
        *   *Benefits:* Reduced feed waste, healthier animals, energy efficiency.

*   **E. AI-Powered Crop Breeding & Genetics**
    *   **1. Accelerated Trait Identification:**
        *   *How AI helps:* Analyzing vast genomic data to identify genes responsible for desirable traits (drought resistance, higher yield, disease immunity).
        *   *Benefits:* Faster development of resilient and productive crop varieties.
        *   *Technologies:* Genomics, machine learning, bioinformatics.

*   **F. Supply Chain Optimization & Food Traceability**
    *   **1. Demand Forecasting & Logistics:**
        *   *How AI helps:* Predicting consumer demand to optimize planting schedules, harvest volumes, and transportation routes.
        *   *Benefits:* Reduced food waste, fresher produce, efficient delivery.
    *   **2. Farm-to-Fork Traceability:**
        *   *How AI helps:* Integrating data from sensors, blockchain, and logistics to provide transparent tracking of produce from its origin to the consumer.
        *   *Benefits:* Enhanced food safety, consumer trust, verification of sustainable practices.

### IV. The Transformative Benefits of AI in Agriculture (Approx. 150-200 words)

*   **A. Increased Productivity & Yields:** More food from less land.
*   **B. Enhanced Sustainability & Environmental Protection:**
    *   Reduced water, fertilizer, and pesticide use.
    *   Lower carbon footprint.
    *   Biodiversity preservation.
*   **C. Improved Food Security:** More stable and predictable food supply globally.
*   **D. Economic Growth for Farmers:** Higher efficiency, lower costs, better market access.
*   **E. Better Quality & Safer Food:** Reduced chemical residues, improved traceability.

### V. Challenges & Ethical Considerations (Approx. 200-250 words)

*   **A. Cost & Accessibility:** High initial investment, digital divide for small farmers.
*   **B. Data Privacy & Ownership:** Who owns the vast amounts of data collected? Potential for misuse.
*   **C. Job Displacement:** Concerns about automation replacing human labor, need for re-skilling.
*   **D. Infrastructure Requirements:** Need for robust internet connectivity (especially in rural areas) and power.
*   **E. AI Bias & Reliability:** Ensuring algorithms are fair, accurate, and robust in diverse conditions.
*   **F. Cybersecurity Risks:** Protecting sensitive farm data and automated systems from cyber threats.
*   **G. Regulatory Frameworks:** Need for clear guidelines and policies for AI adoption in agriculture.

### VI. The Human Element: Farmers as "Agronomists of the Future" (Approx. 100 words)

*   **A. Shifting Roles:** Emphasize that AI isn't replacing farmers but empowering them.
*   **B. New Skill Sets:** The need for data interpretation, technology management, and strategic decision-making.
*   **C. AI as a Tool:** Positioning AI as an intelligent assistant that amplifies human expertise.

### VII. Conclusion (Approx. 150-200 words)

*   **A. Recap Key Points:** Briefly reiterate the immense potential of AI to address agriculture's biggest challenges.
*   **B. Reiterate the Vision:** Paint a picture of a future where agriculture is more efficient, sustainable, resilient, and capable of feeding a growing global population.
*   **C. Call to Action / Forward-Looking Statement:** Encourage continued investment, research, collaboration between tech companies, farmers, and policymakers.
    *   *Example:* "The future of AI in agriculture isn't just about technology; it's about securing a sustainable and prosperous future for humanity. It's a journey that requires collaboration, innovation, and a shared vision for a healthier planet and a well-fed world."
*   **D. Final Thought/Question:** End with a powerful, memorable statement.

---

### Optional Sections:

*   **Glossary of Terms:** For technical terms like IoT, Machine Learning, Computer Vision, LiDAR.
*   **Further Reading/Resources:** Links to reports, organizations, or companies pioneering AI in agriculture.
*   **FAQ Section:**
    *   Is AI too expensive for small farms?
    *   Will robots take all farming jobs?
    *   How secure is farm data?

---

In [14]:
Markdown(final_state['content'])

## Harvesting Tomorrow: How AI is Revolutionizing the Future of Agriculture

By 2050, the world will need to feed nearly 10 billion people. How can we possibly meet this monumental demand while grappling with the escalating challenges of climate change, dwindling arable land, water scarcity, pest resistance, unpredictable weather patterns, and rising labor costs? For centuries, agriculture has been the backbone of human civilization, relying on traditional methods passed down through generations. However, the pressures on modern farming are unprecedented, calling for radical innovation.

Enter Artificial Intelligence. Far from being just a buzzword, AI is emerging as a pivotal solution, ushering in a new era of "smart farming." In this context, AI encompasses a range of technologies, including machine learning, computer vision, robotics, and advanced data analytics, all working in concert to optimize every stage of food production. This blog post will delve into the transformative potential of AI, exploring how it's poised to revolutionize agriculture from farm management to global food security, ensuring a sustainable and prosperous future for humanity.

### The Current Landscape: AI's Footprint in Today's Agriculture

While the full promise of AI in agriculture is still unfolding, its early adoptions are already making a tangible impact. Farmers today utilize basic AI-powered weather forecasting models to plan planting and harvesting schedules. Drones equipped with early mapping capabilities provide initial insights into field health, and automated irrigation systems, while not always complex AI, have laid crucial groundwork for more intelligent resource management. What we see now, however, is merely the tip of the iceberg, hinting at a future far more integrated and intelligent.

### Key Pillars of AI's Future in Agriculture

The true revolution lies in AI's ability to bring unprecedented precision, prediction, and automation to every facet of farming.

#### A. Precision Agriculture & Resource Optimization

One of AI's most profound impacts is in enabling hyper-efficient resource management, minimizing waste and maximizing output.

1.  **Smart Irrigation Systems:** AI analyzes a wealth of data – soil moisture levels from IoT sensors, real-time weather forecasts, specific crop types, and their current growth stages – to deliver precise amounts of water exactly when and where it's needed. This intelligent approach leads to significant water savings, reduces runoff, and optimizes crop health, fostering resilience against drought.
2.  **Variable Rate Application (Fertilizers & Pesticides):** Gone are the days of blanket spraying. Drones and ground-based sensors equipped with spectral imaging collect hyper-local data on plant health, nutrient deficiencies, and pest infestations. AI algorithms then process this data, dictating the exact amounts and specific locations for fertilizer or pesticide application. The benefits are immense: drastically reduced chemical use, lower environmental impact, substantial cost savings for farmers, and ultimately, higher yields of healthier produce.
3.  **Automated Weed Detection & Removal:** Weeds compete fiercely with crops for resources. AI-powered robots, utilizing advanced computer vision and deep learning, can differentiate between crops and weeds with remarkable accuracy. These robots can then either apply targeted micro-doses of herbicides or physically remove weeds mechanically, drastically reducing overall herbicide use and the labor required, leading to healthier crops and ecosystems.

#### B. Predictive Analytics & Crop Monitoring

AI's capacity for predictive analytics is a game-changer, allowing farmers to anticipate and mitigate problems before they escalate.

1.  **Disease and Pest Detection:** Drones, ground sensors, and high-resolution cameras continuously capture images and data from fields. AI models are trained to analyze patterns within this data, identifying the earliest signs of disease or pest infestation – often long before a human eye could detect them. This early intervention capability prevents widespread crop loss and allows for targeted, efficient treatments.
2.  **Yield Prediction & Crop Health Forecasting:** By integrating historical yield data, real-time weather patterns, soil conditions, and a continuous stream of sensor data, AI can accurately predict harvest yields and monitor crop stress levels. This foresight enables better planning for harvest logistics, market pricing, and proactive problem-solving, ensuring a more stable food supply.

#### C. Automated Farming & Robotics

The labor-intensive nature of farming is being transformed by AI-driven automation and robotics.

1.  **Autonomous Tractors & Farm Machinery:** Self-driving tractors, planters, and harvesters are already a reality, performing tasks with minimal human oversight. Equipped with GPS, LiDAR, and computer vision, these machines can operate 24/7, increasing efficiency, reducing labor costs, and executing tasks with unparalleled precision.
2.  **Harvesting & Planting Robots:** From delicate berry-picking robots that identify ripeness using computer vision to precision planting robots that place seeds at optimal depths and spacing, AI-powered robotics are addressing labor shortages, reducing food waste from damaged produce, and improving consistency across fields.

#### D. AI in Livestock Management

AI isn't just for crops; it's revolutionizing animal agriculture too, enhancing welfare and productivity.

1.  **Individual Animal Monitoring:** Wearable sensors and computer vision systems track individual animal health, behavior (e.g., lameness, estrus cycles, feeding patterns), and growth rates. This allows for early detection of illness, improved animal welfare through personalized care, optimized feeding regimens, and increased overall productivity.
2.  **Automated Feeding & Environment Control:** AI-driven systems dispense precise feed portions tailored to individual animal needs, minimizing waste. They also intelligently optimize barn conditions, controlling temperature, humidity, and ventilation to create ideal environments, further improving animal health and energy efficiency.

#### E. AI-Powered Crop Breeding & Genetics

Developing new, resilient crop varieties traditionally takes years. AI is dramatically accelerating this process.

1.  **Accelerated Trait Identification:** AI analyzes vast genomic datasets to quickly identify genes responsible for desirable traits such as drought resistance, higher yield potential, or immunity to specific diseases. This bioinformatics power enables the faster development of more resilient and productive crop varieties, critical for adapting to changing climates.

#### F. Supply Chain Optimization & Food Traceability

Beyond the farm gate, AI is streamlining the entire journey from farm to fork.

1.  **Demand Forecasting & Logistics:** By analyzing consumer buying patterns, market trends, and even social media sentiment, AI can predict consumer demand with greater accuracy. This allows farmers to optimize planting schedules, harvest volumes, and transportation routes, leading to reduced food waste and the delivery of fresher produce.
2.  **Farm-to-Fork Traceability:** Integrating data from sensors, blockchain technology, and logistics platforms, AI provides transparent tracking of produce from its origin to the consumer. This enhances food safety, builds consumer trust by verifying sustainable practices, and helps identify contamination sources quickly.

### The Transformative Benefits of AI in Agriculture

The widespread adoption of AI promises a paradigm shift in how we produce food, yielding multifaceted benefits:

*   **Increased Productivity & Yields:** AI enables farmers to produce more food from less land, utilizing resources more efficiently than ever before.
*   **Enhanced Sustainability & Environmental Protection:** By precisely managing water, fertilizer, and pesticide use, AI drastically reduces agricultural runoff and chemical pollution. It lowers the carbon footprint of farming operations and supports biodiversity preservation.
*   **Improved Food Security:** With more stable, predictable, and resilient food production systems, AI contributes significantly to global food security, reducing instances of scarcity and price volatility.
*   **Economic Growth for Farmers:** Higher efficiency, reduced input costs, and better market access translate into increased profitability and economic stability for farmers.
*   **Better Quality & Safer Food:** Reduced reliance on broad chemical applications and enhanced traceability mean fewer chemical residues and a safer, higher-quality food supply for consumers.

### Challenges & Ethical Considerations

Despite its immense potential, the path to widespread AI adoption in agriculture is not without hurdles:

*   **Cost & Accessibility:** The initial investment in AI technologies can be substantial, creating a digital divide that may exclude small and medium-sized farmers without adequate support.
*   **Data Privacy & Ownership:** The vast amounts of data collected by AI systems raise critical questions about data privacy, ownership, and potential misuse, requiring robust legal and ethical frameworks.
*   **Job Displacement:** Concerns exist about automation replacing human labor, necessitating re-skilling initiatives and new educational pathways to prepare the agricultural workforce for evolving roles.
*   **Infrastructure Requirements:** Reliable, high-speed internet connectivity and consistent power supply are essential for AI systems, posing significant challenges in many rural and remote farming regions.
*   **AI Bias & Reliability:** Ensuring that algorithms are fair, accurate, and robust across diverse agricultural conditions, soil types, and climates is crucial to prevent unintended consequences or suboptimal outcomes.
*   **Cybersecurity Risks:** Protecting sensitive farm data and automated systems from cyber threats and malicious attacks will become increasingly vital as farms become more interconnected.
*   **Regulatory Frameworks:** The rapid pace of AI innovation demands clear and adaptive regulatory frameworks to guide its responsible development and deployment in agriculture.

### The Human Element: Farmers as "Agronomists of the Future"

It's crucial to understand that AI isn't here to replace farmers; it's here to empower them. The role of the farmer is shifting from manual labor to that of a sophisticated "agronomist of the future." Farmers will increasingly become data interpreters, technology managers, and strategic decision-makers, leveraging AI as an intelligent assistant. This transformation requires new skill sets, focusing on understanding AI insights, managing automated systems, and making informed choices that amplify human expertise and intuition.

### Conclusion

The future of AI in agriculture is not a distant dream but a rapidly unfolding reality. From revolutionizing precision farming and predictive analytics to enabling automated robotics and enhancing food traceability, AI offers an unparalleled opportunity to address some of humanity's most pressing challenges. It promises a future where agriculture is more efficient, sustainable, resilient, and capable of feeding a growing global population without compromising our planet.

This transformative journey, however, requires more than just technological advancement. It demands continued investment in research and development, robust infrastructure development, thoughtful policy-making, and proactive collaboration between tech innovators, farmers, governments, and educational institutions. The future of AI in agriculture isn't just about technology; it's about securing a sustainable and prosperous future for humanity. It's a journey that requires collaboration, innovation, and a shared vision for a healthier planet and a well-fed world.

Are we ready to cultivate this future?

In [ ]:
# One great thing is we can access the intermediate state after the first node to see the generated outline before proceeding to content generation. This allows for better debugging and understanding of the workflow.